<a href="https://colab.research.google.com/github/aravindhan-22/hospital-readmission-ml-model/blob/main/notebooks/hospital_readmission_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import pandas as pd

# Load dataset from GitHub
url = "https://raw.githubusercontent.com/aravindhan-22/hospital-readmission-ml-model/main/dataset/hospital_patients.csv"

patients = pd.read_csv(url, sep="\t")

# Check data
patients.head()

,age,gender,diagnosis,treatment,days_in_hospital,num_previous_admissions,readmitted
0,45,M,Diabetes,Insulin,4,1,Yes
1,67,F,Heart Disease,Surgery,7,3,Yes
2,31,M,Pneumonia,Antibiotics,3,0,No
3,52,F,Diabetes,Insulin,5,2,Yes
4,unknown,M,Heart Disease,Surgery,6,1,No


In [51]:
print(patients.shape)


(30, 7)


In [52]:
print(patients.columns)


Index(['age', 'gender', 'diagnosis', 'treatment', 'days_in_hospital',
       'num_previous_admissions', 'readmitted'],
      dtype='object')


In [53]:
print(patients.isnull().sum())


age                        0
gender                     0
diagnosis                  3
treatment                  0
days_in_hospital           0
num_previous_admissions    0
readmitted                 0
dtype: int64


In [54]:
print(patients[patients['age'] == 'unknown'])


        age gender      diagnosis treatment  days_in_hospital  \
4   unknown      M  Heart Disease   Surgery                 6   
24  unknown      F  Heart Disease   Surgery                 5   

    num_previous_admissions readmitted  
4                         1         No  
24                        2        Yes  


In [55]:
print(patients.dtypes)


age                        object
gender                     object
diagnosis                  object
treatment                  object
days_in_hospital            int64
num_previous_admissions     int64
readmitted                 object
dtype: object


In [56]:
patients['age'] = patients['age'].replace('unknown', pd.NA)

In [57]:
patients['age'] = pd.to_numeric(patients['age'])


In [58]:
average_age = patients['age'].mean()
patients['age'] = patients['age'].fillna(round(average_age))

In [59]:
print(patients['age'])
print("Data type now:", patients['age'].dtype)

0     45.0
1     67.0
2     31.0
3     52.0
4     54.0
5     78.0
6     43.0
7     55.0
8     61.0
9     49.0
10    38.0
11    72.0
12    44.0
13    58.0
14    66.0
15    39.0
16    51.0
17    63.0
18    47.0
19    70.0
20    35.0
21    80.0
22    29.0
23    57.0
24    54.0
25    42.0
26    68.0
27    33.0
28    75.0
29    50.0
Name: age, dtype: float64
Data type now: float64


In [60]:
print(patients[patients['diagnosis'].isnull()])


     age gender diagnosis    treatment  days_in_hospital  \
5   78.0      F       NaN  Antibiotics                 2   
14  66.0      M       NaN  Antibiotics                 5   
25  42.0      M       NaN  Antibiotics                 3   

    num_previous_admissions readmitted  
5                         4        Yes  
14                        3         No  
25                        1         No  


In [61]:
print(patients['diagnosis'].value_counts())


diagnosis
Diabetes         10
Heart Disease     9
Pneumonia         8
Name: count, dtype: int64


In [62]:
patients['diagnosis'] = patients['diagnosis'].fillna('Diabetes')


In [63]:
print(patients['diagnosis'].isnull().sum())


0


In [64]:
print(patients.isnull().sum())
print("---")
print(patients.dtypes)

age                        0
gender                     0
diagnosis                  0
treatment                  0
days_in_hospital           0
num_previous_admissions    0
readmitted                 0
dtype: int64
---
age                        float64
gender                      object
diagnosis                   object
treatment                   object
days_in_hospital             int64
num_previous_admissions      int64
readmitted                  object
dtype: object


In [65]:
print(patients.describe())


             age  days_in_hospital  num_previous_admissions
count  30.000000         30.000000                30.000000
mean   53.533333          5.066667                 2.033333
std    14.284627          1.946408                 1.670914
min    29.000000          2.000000                 0.000000
25%    43.250000          3.250000                 1.000000
50%    53.000000          5.000000                 2.000000
75%    65.250000          6.750000                 3.000000
max    80.000000          9.000000                 6.000000


In [66]:
print(patients.groupby('readmitted')['num_previous_admissions'].mean())


readmitted
No     0.916667
Yes    2.777778
Name: num_previous_admissions, dtype: float64


In [67]:
print(patients['readmitted'].value_counts())


readmitted
Yes    18
No     12
Name: count, dtype: int64


In [68]:
from sklearn.preprocessing import LabelEncoder

# Convert text columns to numbers
le = LabelEncoder()
patients['gender_enc']    = le.fit_transform(patients['gender'])
patients['diagnosis_enc'] = le.fit_transform(patients['diagnosis'])
patients['treatment_enc'] = le.fit_transform(patients['treatment'])
patients['readmitted_enc']= le.fit_transform(patients['readmitted'])

# Separate features and target
X = patients[['age', 'gender_enc', 'diagnosis_enc',
              'treatment_enc', 'days_in_hospital',
              'num_previous_admissions']]
y = patients['readmitted_enc']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (30, 6)
Target shape: (30,)


In [69]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Training patients:", len(X_train))
print("Testing patients:", len(X_test))

# Train the model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")

Training patients: 24
Testing patients: 6
Model trained successfully!


In [70]:
from sklearn.metrics import accuracy_score

# Make predictions on test patients
predictions = model.predict(X_test)

# Check accuracy
accuracy = accuracy_score(y_test, predictions)
print("Model Accuracy:", round(accuracy * 100), "%")

# See actual vs predicted
print("\nActual vs Predicted:")
for actual, predicted in zip(y_test, predictions):
    result = "✅" if actual == predicted else "❌"
    print(f"Actual: {actual}  Predicted: {predicted}  {result}")

Model Accuracy: 83 %

Actual vs Predicted:
Actual: 0  Predicted: 0  ✅
Actual: 1  Predicted: 1  ✅
Actual: 1  Predicted: 1  ✅
Actual: 1  Predicted: 1  ✅
Actual: 0  Predicted: 0  ✅
Actual: 0  Predicted: 1  ❌


In [71]:
import pandas as pd

# Get feature importances
feature_names = ['age', 'gender', 'diagnosis',
                 'treatment', 'days_in_hospital',
                 'num_previous_admissions']

importances = pd.Series(
    model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("Feature Importances:")
print(importances)

Feature Importances:
num_previous_admissions    0.233648
diagnosis                  0.197459
gender                     0.185117
age                        0.182540
days_in_hospital           0.119689
treatment                  0.081548
dtype: float64


In [72]:
# New patient — 72 years old, Female, Diabetes,
# Insulin treatment, 7 days in hospital, 5 previous admissions
new_patient = [[72, 1, 0, 1, 7, 5]]

prediction = model.predict(new_patient)
probability = model.predict_proba(new_patient)

print("Prediction:", "Readmitted" if prediction[0]==1 else "Not Readmitted")
print("Probability of readmission:", round(probability[0][1]*100), "%")

Prediction: Readmitted
Probability of readmission: 85 %


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
